In [1]:
import sys
import os

# Get the parent directory (project root)
# this assumes notebooks is a dir within root dir
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)



In [2]:
### on myriad, the prefix of this dir means it can't be used in other file paths
### i don't know what that means
### but don't get confused with other file paths set below
print(project_root)

## for reloading functions before running. helps when developing code
%load_ext autoreload
%autoreload 2

/myriadfs/home/ucbtvsi/Image-Analysis-Summer-Project


In [3]:
from scripts.load_images import load_images
from scripts.segment import save_mask_overlay, segment_frames, estimate_diameter, validate_segmentation_objects
from scripts.track import extract_centroids, track_particles
from scripts.utils import plot_size_distribution

import yaml




Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	linux 
python version: 	3.10.18 
torch version:  	2.7.1! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 




In [4]:
config_file = '/home/ucbtvsi/Image-Analysis-Summer-Project/configs/params.yaml' #copy the path from your computer

with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)
    print('config loaded...')

#### so you know what's happening
testing = cfg['testing']
downsize_size = cfg['downsize_size']
frame_subset = cfg['frame_subset']
subset_frame_number = cfg['subset_frame_number']
image_dir = cfg['image_folder']
outdir = cfg['output_dir']
print('Some things for you to check before proceeding:')
print('testing is set to ' + str(testing))
print('if testing is true, we will downsize images to ' + str(downsize_size) + ' to improve performance')
print('using a subset of frames is ' + str(frame_subset))
print( 'if using a subset of frames is true, we will use ' + str(subset_frame_number) + ' frames' )

config loaded...
Some things for you to check before proceeding:
testing is set to True
if testing is true, we will downsize images to [50, 50] to improve performance
using a subset of frames is True
if using a subset of frames is true, we will use 10 frames


In [5]:
frames_in, tif_files = load_images(image_dir, testing=testing)
if frames_in.any():
    print('***images loaded***')
else:
    print('***warning: no images loaded***')
    
# may only want to run on a subset of frames
if frame_subset:
    print("subsetting to use " + str(cfg['subset_frame_number']) + " frames")
    frames = frames_in[:cfg['subset_frame_number']]
else:
    print( "using all " + str(len(frames_in)) + " frames")
    frames = frames_in

Found 200 TIFF files.
Downsizing frames to (100, 100) for testing...
***images loaded***
subsetting to use 10 frames


In [6]:
masks = segment_frames(
    frames,
    gpu = cfg.get('gpu', False),
    output_dir= outdir,
    save_overlays=True,
    file_names=[os.path.basename(f) for f in tif_files]  # optional
)

segmenting image IXMtest_A02_s1_w1051DAA7C-7042-435F-99F0-1E847D9B42CB.tif
saving mask for image IXMtest_A02_s1_w1051DAA7C-7042-435F-99F0-1E847D9B42CB.tif
segmenting image IXMtest_A06_s6_w1B9577918-4973-4A87-BA73-A168AA755527.tif
saving mask for image IXMtest_A06_s6_w1B9577918-4973-4A87-BA73-A168AA755527.tif
segmenting image IXMtest_A09_s1_w1CE70AD49-290D-4312-82E6-CDC717F32637.tif
saving mask for image IXMtest_A09_s1_w1CE70AD49-290D-4312-82E6-CDC717F32637.tif
segmenting image IXMtest_A12_s7_w1EAEEA614-51ED-43B3-A4FF-088730911E4C.tif
saving mask for image IXMtest_A12_s7_w1EAEEA614-51ED-43B3-A4FF-088730911E4C.tif
segmenting image IXMtest_A15_s5_w1825174D4-ED30-490C-9635-6196417D6C9D.tif
saving mask for image IXMtest_A15_s5_w1825174D4-ED30-490C-9635-6196417D6C9D.tif
segmenting image IXMtest_A16_s2_w15AF20A10-82AE-48FA-AC50-7AE8AC3AA544.tif
saving mask for image IXMtest_A16_s2_w15AF20A10-82AE-48FA-AC50-7AE8AC3AA544.tif
segmenting image IXMtest_A16_s3_w1032BE329-E21B-4E1B-B4B8-58700685EE0C

In [ ]:
##### some stuff about validating

# 1. check boundaries

# 2. check masks

# 3. check diameters

# 4. size histograms - are there any outliers? Are these mistakes?





In [7]:

### first let's look at some frames in more detail.
validate_segmentation_objects(
    frames,
    masks,
    file_names=tif_files,
    output_dir= outdir,
    sample_fraction=cfg.get('validate_sample_fraction', 0.05),
    min_samples=cfg.get('validate_min_samples', 5)
)



QC images saved in: /home/ucbtvsi/Image-Analysis-Summer-Project/output/validation/object_labels
ROI table saved at: /home/ucbtvsi/Image-Analysis-Summer-Project/output/validation/object_table.csv


In [8]:
### Now we are confident (or not :'D )the segmentation and size estimates are generally correct
### let's look at the overall distribution

## we plot the distribution of all objects
## small outliers might suggest small fragments have been classified as objects
## large outliers might indicate poor separation of close objects
plot_size_distribution(
    masks=masks,
    file_names=tif_files,
    output_dir=outdir
)

Size distribution plot saved to: /home/ucbtvsi/Image-Analysis-Summer-Project/output/validation/size_distribution.png
Summary stats saved to: /home/ucbtvsi/Image-Analysis-Summer-Project/output/validation/size_distribution_stats.csv


In [ ]:
# for the demo, it would be good to have some other examples
# we should download some other images (maybe some tricky ones?) and segment those too
# for each set of images, you will just need to update the following and re run the code (in theory...)
# testing = cfg['testing']
# frame_subset = cfg['frame_subset']
# subset_frame_number = cfg['subset_frame_number']
# image_dir = cfg['image_folder']
# outdir = cfg['outdir']